# 04 — Pipeline données (traitement complet)

**Emplacement** : `01_donnees/04_pipeline_donnees.ipynb`

| Étape | Contenu |
|-------|--------|
| 1 | Inventaire des CSV sources |
| 2 | Rapport qualité (voir aussi `02_qualite_hallucinations.ipynb`) |
| 3 | Vérification registre RGPD (`03_registre_rgpd.ipynb`) |
| 4 | Pipeline : coffre identité → cohorte → features |
| 5 | Contrôles des exports `data/curated/` |

Équivalent terminal : `PYTHONPATH=. python scripts/run_pipeline.py`

Navigation : [`notebooks/README.md`](../README.md)


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.load import load_csv
from src.data.paths import CSV_FILES, ProjectPaths, get_project_root
from src.data.pipeline import run_pipeline
from src.data.quality_report import export_quality_report, run_quality_report
from src.data.registre import (
    assert_justifications,
    assert_registre_covers_sources,
    build_source_column_index,
    load_registre,
)

paths = ProjectPaths(root=get_project_root())
print("Racine projet :", ROOT)

## 1. Inventaire des sources

Grain, clés, cardinalités — **sans nettoyage**.

In [ ]:
inventaire = []
for name in CSV_FILES:
    df = load_csv(paths, name, from_raw=False)
    inventaire.append({
        "fichier": name,
        "lignes": len(df),
        "colonnes": len(df.columns),
        "aperçu_clés": ", ".join(df.columns[:4].tolist()) + ("…" if len(df.columns) > 4 else ""),
    })
inv_df = pd.DataFrame(inventaire)
display(inv_df)

## 2. Rapport qualité (graphiques par jeu de données)

Pour chaque CSV : **trous** (manquants), **valeurs aberrantes** (hors plage), **mal notées** (incohérences), **sensibilité & usage IA** (registre, ex. `actes.csv`).

In [ ]:
qualite_out = export_quality_report(paths)
print("Rapport HTML :", qualite_out["html"])
print("Rapport JSON :", qualite_out["json"])

report_qualite = run_quality_report(paths)
resume = []
for ds in report_qualite["datasets"]:
    resume.append({
        "fichier": ds["fichier"],
        "trous_max": round(max(ds["trous"].values()) if ds["trous"] else 0, 3),
        "aberrants": len(ds["aberrants"]),
        "mal_notes": len(ds["mal_notes"]),
        "colonnes_IA": len(ds["sensibilite_ia"]),
    })
display(pd.DataFrame(resume))
display(HTML(f'<p><a href="../data/curated/qualite/rapport_qualite.html" target="_blank">Ouvrir le rapport HTML complet (graphiques)</a></p>'))

## 3. Registre de sensibilité (RGPD)

Chaque colonne des CSV sujet doit être cataloguée ; les variables sensibles utilisées par l'IA doivent être justifiées.

In [ ]:
registre = load_registre(paths.registres / "registre_colonnes.csv")
index = build_source_column_index(paths)
assert_registre_covers_sources(registre, index)
assert_justifications(registre)
print(f"Registre OK — {len(registre)} entrées")

# Exemple actes.csv : usage IA
actes_reg = registre.loc[registre["fichier_source"] == "actes.csv"]
display(actes_reg[["colonne", "sensibilite", "usage_score_sortie", "usage_score_tele", "risque_principal"]])

## 4. Pipeline complet

Copie raw → coffre identité → pseudonymisation → qualité → cohorte Domicile → features score sortie & télé.

In [ ]:
out = run_pipeline(paths, horizon_tele=7)
for k, p in out.items():
    print(f"{k}: {p}")

## 5. Vérification des exports

Contrôles : pas de nominatif dans curated, objets connectés seulement dans score télé, cible présente.

In [ ]:
rapport = json.loads(out["rapport"].read_text(encoding="utf-8"))
fs = pd.read_parquet(out["features_score_sortie"])
ft = pd.read_parquet(out["features_score_tele"])

assert "NomPrenom" not in fs.columns
assert "NomPrenom" not in ft.columns
assert not any(c.startswith("oc_") for c in fs.columns)
assert any(c.startswith("oc_") for c in ft.columns)
assert "Readmission30j" in fs.columns

print("Séjours éligibles :", rapport["n_sejours_eligibles"])
print("Prévalence réadmission :", f"{rapport['prevalence_readmission']:.1%}")
print("Features sortie :", rapport["n_features_sortie"], "· télé :", rapport["n_features_tele"])
print("\nÉtape suivante → notebook `pipeline_entrainement.ipynb`")